# Data Wrangling — Falcon 9 First-Stage Landing Dataset

This notebook documents the cleaning and feature-preparation steps for the frozen IBM course Falcon 9 dataset. It preserves the course Class target (1 = successful first-stage landing, 0 = unsuccessful first-stage landing).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA=Path('data/dataset_part_2.csv')
OUT=Path('outputs/wrangling')
OUT.mkdir(parents=True,exist_ok=True)
if not DATA.exists(): raise FileNotFoundError(DATA)
df=pd.read_csv(DATA)
df.columns=[str(c).strip() for c in df.columns]
required={'FlightNumber','Date','PayloadMass','Orbit','LaunchSite','Class'}
missing=required-set(df.columns)
if missing: raise ValueError(f'Missing required fields: {sorted(missing)}')
print('Initial shape:',df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame('Missing values'))

## 1. Validate data types and target definition

Convert dates and payload mass to appropriate types. Records without a valid target are excluded; unknown landing outcomes are not guessed.

In [ ]:
df['Date']=pd.to_datetime(df['Date'],errors='coerce')
df['FlightNumber']=pd.to_numeric(df['FlightNumber'],errors='coerce')
df['PayloadMass']=pd.to_numeric(df['PayloadMass'],errors='coerce')
df['Class']=pd.to_numeric(df['Class'],errors='coerce')
df=df.dropna(subset=['FlightNumber','Date','Class']).copy()
df['Class']=df['Class'].astype(int)
if not set(df['Class'].unique()).issubset({0,1}): raise ValueError('Class must be binary 0/1')
print('Validated shape:',df.shape)
display(df.dtypes.to_frame('Data type'))

## 2. Treat missing payload mass

Payload mass is numeric. For this exercise, missing payload values are replaced with the observed mean. The imputation value is printed and retained for transparency.

In [ ]:
payload_mean=df['PayloadMass'].mean()
missing_payload=int(df['PayloadMass'].isna().sum())
df['PayloadMass']=df['PayloadMass'].fillna(payload_mean)
print('Missing payload values before imputation:',missing_payload)
print('Imputation mean (kg):',round(float(payload_mean),3))
print('Missing payload values after imputation:',int(df['PayloadMass'].isna().sum()))

## 3. Select model features and encode categorical variables

Use the course feature set where available. One-hot encode categorical fields and preserve the Class target separately. Avoid including raw outcome text or the Class label among the predictors.

In [ ]:
candidate_features=['FlightNumber','PayloadMass','Orbit','LaunchSite','Flights','GridFins','Reused','Legs','LandingPad','Block','ReusedCount','Serial']
features=[c for c in candidate_features if c in df.columns]
if not features: raise ValueError('No expected model feature columns found')
X=df[features].copy()
y=df['Class'].copy()
categorical=X.select_dtypes(include=['object','category','string']).columns.tolist()
X_encoded=pd.get_dummies(X,columns=categorical,dtype=float)
X_encoded=X_encoded.replace([np.inf,-np.inf],np.nan)
for c in X_encoded.columns:
    if X_encoded[c].isna().any(): X_encoded[c]=X_encoded[c].fillna(X_encoded[c].median())
print('Selected base features:',features)
print('Categorical variables encoded:',categorical)
print('Encoded feature matrix shape:',X_encoded.shape)
print('Target shape:',y.shape)
display(X_encoded.head())

## 4. Export and quality checks

The official course dataset_part_3.csv remains the reference feature matrix used by the machine-learning exercise. The recreated file is exported for comparison, not silently substituted for the official feature file.

In [ ]:
df.to_csv(OUT/'dataset_part_2_cleaned.csv',index=False)
X_encoded.to_csv(OUT/'dataset_part_3_recreated.csv',index=False)
y.to_csv(OUT/'landing_target_class.csv',index=False,header=['Class'])
quality=pd.DataFrame({
    'Rows':[len(df)],
    'Base feature count':[len(features)],
    'Encoded feature count':[X_encoded.shape[1]],
    'Missing payload after imputation':[int(df['PayloadMass'].isna().sum())],
    'Missing target after cleaning':[int(df['Class'].isna().sum())],
    'Duplicate rows':[int(df.duplicated().sum())]
})
display(quality)
print('Outputs saved under:',OUT.resolve())